# 01. Measles TCR-beta analysis

Authoritative downstream analysis. Clinical timepoints come from curated metadata. Age split: child < 6 years; adult >= 6 years. Final biological QC requires >=5,000 productive TRB clonotypes per library.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path.cwd().resolve()
os.environ['MEASLES_REPO'] = str(ROOT)
print('Notebook root:', ROOT)
def run_script(name):
    path = ROOT / 'scripts' / name
    print('RUN', path)
    exec(compile(path.read_text(encoding='utf-8'), str(path), 'exec'), globals())


## 1. Build biological samples and normalize repertoire depth


In [ ]:
run_script('analysis_01_prepare.py')


## 2. Exact specificity-reference audit


In [ ]:
run_script('analysis_02_antigens.py')


## 3. Raw FASTQ depth QC


In [ ]:
raw_depth = ROOT/'results'/'raw_fastq_depth.csv'
if raw_depth.exists() and raw_depth.stat().st_size > 0:
    print('Reuse', raw_depth)
else:
    subprocess.run([sys.executable, str(ROOT/'scripts'/'collect_fastq_depth.py'), '--workers', '8'], check=True)


## 4. Clinically corrected base analysis


In [ ]:
run_script('analysis_06_clinical_summary.py')


## 5. Annotated age-6 reporting


In [ ]:
run_script('analysis_07_age6_reporting.py')


## 6. Final QC-filtered reporting

This is the authoritative figure set. It applies the >=5,000 productive-clonotype threshold and rebuilds patient figures, including explicit QC display for CA1_013.

In [ ]:
run_script('analysis_08_final_reporting.py')


In [ ]:
required = [
    ROOT/'results'/'current'/'sample_metadata_current.csv',
    ROOT/'results'/'current'/'cross_sectional_statistics_current.csv',
    ROOT/'results'/'current'/'T1_T2_paired_statistics_current.csv',
    ROOT/'results'/'current'/'venn_T1_T2_vs_healthy_current.csv',
    ROOT/'figures'/'current'/'00_library_QC_overview.png',
    ROOT/'figures'/'current'/'01_oligoclonality_case_control.png',
    ROOT/'figures'/'current'/'05_T1_T2_richness.png',
    ROOT/'figures'/'current'/'07_venn_and_weighted_similarity_summary.png',
    ROOT/'figures'/'current'/'08_TRBV_usage_adult.png',
    ROOT/'figures'/'current'/'09_TRBJ_usage_adult.png',
    ROOT/'figures'/'current'/'patients'/'СА1_013_repertoire_QC_and_overlap.png',
]
missing=[str(p) for p in required if not p.exists() or p.stat().st_size==0]
if missing: raise RuntimeError(f'Final analysis missing required outputs: {missing}')
print('FINAL current outputs verified:', ROOT/'figures'/'current')
